# E1 · Limpieza y preprocesamiento — Observatorio del Comercio al por Menor en Colima

Flujo: **Bronce** (JSON crudo de INEGI y Banxico) → **integración** → **Puerta de Calidad (Pandera)** → **Plata** → **limpieza estadística** → **Oro**.

Este notebook usa las funciones de `src/` (las mismas que ejecuta GitHub Actions), así que lo que se ve aquí es exactamente lo que corre en automático.

> Si todavía no hay capturas reales en `E0/data/bronze`, el notebook usa una **Bronce sintética** con defectos sembrados (`src/demo_datos.py`). Antes de entregar, ejecútenlo de nuevo con los datos reales y guárdenlo **con salidas**.

In [ ]:
import json, sys, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ_E1 = Path.cwd()
while not (RAIZ_E1 / "src" / "pipeline.py").exists() and RAIZ_E1 != RAIZ_E1.parent:
    RAIZ_E1 = RAIZ_E1.parent
sys.path.insert(0, str(RAIZ_E1))

from src import config, integracion, calidad, limpieza, pipeline
from src.demo_datos import generar

R = config.rutas()
hay_datos_reales = any(any((R.bronce / f).glob("*/*.json")) for f in ("inegi", "banxico"))
if hay_datos_reales:
    BRONCE, SALIDA, MODO = R.bronce, None, "REAL"
else:
    BRONCE = generar(Path(tempfile.mkdtemp()) / "bronce")
    SALIDA, MODO = RAIZ_E1 / "data_demo", "DEMO (datos sintéticos)"
R = config.rutas(BRONCE, SALIDA).crear()
pd.set_option("display.max_columns", 40, "display.width", 160)
plt.rcParams["figure.figsize"] = (11, 3.6)
print("Modo:", MODO, "| Bronce:", BRONCE)

## 1. Integración de fuentes heterogéneas y resolución de conflictos

| Aspecto | INEGI (BIE) | Banxico (SIE) | Resolución |
|---|---|---|---|
| Esquema JSON | `Series[0].OBSERVATIONS[{TIME_PERIOD, OBS_VALUE}]` | `bmx.series[0].datos[{fecha, dato}]` | Un lector por fuente → formato largo común |
| Fecha | `'2024/03'` | `'15/03/2024'` (día/mes/año) | `datetime` al día 1 del mes |
| Valor | texto | texto con coma de miles y `'N/E'` | `float`; el texto original se conserva en `valor_crudo` |
| Frecuencia | mensual | diaria | promedio mensual (reducción por agregación) |
| Revisiones | cifras preliminares cambian entre capturas | — | gana la captura más reciente; el conflicto se registra |
| Nombres | clave de indicador | clave de serie | catálogo `config.SERIES` → (entidad, variable, unidad) |

In [ ]:
crudo = integracion.leer_bronce(BRONCE)
print("Registros crudos por fuente:"); display(crudo.groupby("fuente").size().to_frame("registros"))
display(crudo.groupby("fuente").head(2)[["serie", "fuente", "periodo_crudo", "valor_crudo", "archivo"]])

largo, info = integracion.integrar(BRONCE)
print(json.dumps(info["resumen"], indent=1, ensure_ascii=False))
print("Conflictos de revisión (misma serie y periodo, valor distinto entre capturas):")
display(info["conflictos"])
display(largo.sample(5, random_state=1))

## 2. Puerta de Calidad automática (Pandera) y cuarentena

Reglas (`src/calidad.py`):
- **Esquema estricto**: exactamente las columnas esperadas, con tipo; si falta o sobra una columna, la Puerta se cierra y no se escribe Plata.
- `serie`, `fuente`, `entidad` y `variable` dentro de su catálogo; `fecha` entre 2005 y hoy; sin vacíos en campos obligatorios.
- `valor` dentro del **rango plausible de su variable** (`config.RANGOS`).
- `valor_crudo` debe ser número **o** un faltante declarado por la fuente (`'N/E'`): el texto basura no se confunde con un faltante.
- Unidad consistente con el catálogo y **un solo registro por serie y mes**.

Las filas que fallan van a `data/cuarentena/cuarentena.csv` con **motivo, regla, columna, valor que falló y hora de la corrida**. El archivo es *append-only*: nunca se borra nada en silencio.

In [ ]:
print(calidad.construir_esquema())
validos, cuarentena, resumen_puerta = calidad.aplicar_puerta(largo, R)
print(json.dumps(resumen_puerta, indent=1, ensure_ascii=False))
display(cuarentena[["serie", "fecha", "valor_crudo", "motivo", "columna", "corrida_utc"]])

### 2.1 Comprobación: ningún registro desaparece

In [ ]:
assert resumen_puerta["registros_evaluados"] == resumen_puerta["registros_validos"] + resumen_puerta["registros_en_cuarentena"]
print("OK: evaluados = válidos + cuarentena")

## 3. Valores faltantes: diagnóstico del mecanismo e imputación

- **MCAR** (completamente al azar): la falta no se relaciona con nada observado.
- **MAR** (al azar condicional): la falta se explica con otras variables observadas o con el calendario.
- **MNAR** (no al azar): la falta depende del propio valor no observado. No se puede probar con los datos; se argumenta por el proceso de la fuente (p. ej., INEGI suprime una cifra por confidencialidad cuando hay muy pocas empresas).

Pruebas: Mann-Whitney (¿los meses con falta tienen valores distintos en otras series?) y chi-cuadrada contra el mes del año, con corrección de Bonferroni.

In [ ]:
ancho = pipeline.a_ancho(validos)
faltantes = limpieza.diagnosticar_faltantes(ancho)
display(faltantes)

fig, ax = plt.subplots(figsize=(11, 3))
ax.imshow(ancho.isna().T, aspect="auto", interpolation="nearest", cmap="Greys")
ax.set_yticks(range(len(ancho.columns)), ancho.columns)
ticks = np.arange(0, len(ancho), 24)
ax.set_xticks(ticks, [ancho.index[i].strftime("%Y") for i in ticks])
ax.set_title("Mapa de faltantes (negro = faltante)"); plt.tight_layout(); plt.show()

**Decisión de imputación** (`limpieza.imputar`):
- Solo se imputan huecos **interiores**. Los del final (rezago de publicación) no se imputan: son justo lo que E1 va a pronosticar. Los del inicio son meses en que la fuente aún no existía.
- Huecos de **≤ 3 meses** → interpolación lineal en el tiempo (las series son suaves mes a mes).
- Huecos **más largos** → mismo mes del año anterior escalado por el nivel reciente, para no borrar la estacionalidad.
- Cada celda queda etiquetada con su método (`reportes/metodo_por_celda.csv`).

In [ ]:
limpio, detalle = pipeline.limpiar(ancho)
display(detalle["metodos"].stack().value_counts().to_frame("celdas"))

col = "personal_colima" if "personal_colima" in ancho else ancho.columns[0]
m = detalle["metodos"][col]
fig, ax = plt.subplots()
ax.plot(limpio.index, limpio[col], lw=1, label="serie limpia")
imp = m.isin(["interpolacion_lineal", "estacional_escalada"])
ax.scatter(limpio.index[imp], limpio.loc[imp, col], color="C3", zorder=3, label="imputado")
ax.set_title(f"{col}: valores imputados"); ax.legend(); plt.tight_layout(); plt.show()

## 4. Atípicos y ruido: Z-score robusto, IQR e Isolation Forest

Se analiza la **variación mensual en logaritmos sin su patrón estacional**; así la tendencia y la caída normal de enero no se confunden con atípicos.

- **Z-score robusto** = 0.6745·(x − mediana)/MAD, umbral 3.5 (Iglewicz y Hoaglin). No se deja arrastrar por el mismo atípico, a diferencia del Z-score clásico.
- **IQR** (1.5×) como segundo criterio.
- **Isolation Forest** para atípicos *multivariados*: meses en que la combinación ingresos-personal-remuneraciones es rara.

**Tratamiento**: un *pico aislado* (sube y regresa al mes siguiente) fuera de un evento conocido se considera **ruido de captura** y se reemplaza por interpolación. Los saltos de 2020 (cierre por COVID-19) son **información real**: se conservan y solo se marcan. El valor original siempre queda en `reportes/atipicos.csv`.

In [ ]:
at = detalle["atipicos"]
display(at.groupby(["serie", "tratamiento"]).size().unstack(fill_value=0))

col = "ingresos_colima" if "ingresos_colima" in ancho else ancho.columns[0]
a = at[at.serie.eq(col)].set_index("fecha")
fig, ax = plt.subplots()
ax.plot(ancho.index, ancho[col], lw=1, color="0.6", label="Plata (original)")
ax.plot(limpio.index, limpio[col], lw=1.3, label="limpia")
for trat, color in [("reemplazado_por_interpolacion", "C3"), ("conservado_evento_conocido", "C2")]:
    f = a.index[a["tratamiento"].eq(trat)]
    ax.scatter(f, ancho.loc[f, col], color=color, zorder=3, label=trat)
ax.set_title(f"{col}: atípicos detectados y tratamiento"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
tablas, oro_info = pipeline.construir_oro(limpio)
panel = tablas["panel_colima_mensual"].set_index("fecha")
print("Meses atípicos multivariados (Isolation Forest):", oro_info["meses_atipico_multivariado"])

## 5. Normalización (Min-Max) y estandarización (Z-score)

- **Min-Max** lleva cada variable a [0, 1]: útil para comparar series en un mismo gráfico y para modelos sensibles a la escala.
- **Z-score** centra en 0 con desviación 1: útil para regresión y para la MANOVA de E3.
- Los parámetros se calculan **solo con el periodo de entrenamiento** (todo menos los últimos 12 meses), para no filtrar información del periodo de prueba del pronóstico de E1. Se guardan en `oro/parametros_escalado.json`.

In [ ]:
print("Fin del entrenamiento:", oro_info["fin_entrenamiento"])
display(pd.DataFrame(oro_info["parametros_escalado"]).T)
mm = [c for c in panel.columns if c.endswith("_minmax")]
zz = [c for c in panel.columns if c.endswith("_z")]
fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
panel[mm].plot(ax=axs[0], lw=1, title="Min-Max [0, 1]")
panel[zz].plot(ax=axs[1], lw=1, title="Z-score")
for a_ in axs: a_.axvline(pd.Timestamp(oro_info["fin_entrenamiento"]), ls="--", color="k", lw=0.8); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 6. Discretización y jerarquías conceptuales

| Variable discreta | Regla | Uso |
|---|---|---|
| `temporada` | alta = julio, agosto, noviembre y diciembre | factor A de E2 |
| `tiie_nivel` | alta/baja respecto a la mediana de entrenamiento | factor B de E2 |
| `inflacion_nivel` | alta/baja respecto a la mediana de la inflación anual | factor C de E2 |
| `crecimiento_ingresos` | terciles del crecimiento anual: bajo / medio / alto | descripción y clasificación |

Jerarquías: **tiempo** mes → trimestre → semestre → año; **geografía** entidad → región (Occidente) → país; **sector** subsector 46 → comercio al por menor → terciario.

In [ ]:
cols = [c for c in ["mes", "trimestre", "semestre", "anio", "temporada", "tiie_nivel", "inflacion_nivel", "crecimiento_ingresos"] if c in panel]
display(panel[cols].tail(6))
if {"temporada", "tiie_nivel", "ingresos_crec_anual"} <= set(panel.columns):
    print("Crecimiento anual promedio de ingresos (%) por celda del diseño 2x2 de E2:")
    display(panel.pivot_table(index="temporada", columns="tiie_nivel", values="ingresos_crec_anual", aggfunc="mean").round(2))
if "ingresos_colima" in panel:
    print("Roll-up de la jerarquía de tiempo (mes -> año):")
    display(panel.groupby("anio")["ingresos_colima"].mean().round(2).to_frame("ingresos promedio").T)

## 7. Reducción de datos

1. **Agregación**: los datos diarios de Banxico se reducen a promedios mensuales.
2. **Ventana temporal**: Oro usa desde `2015-01`, periodo común a todas las fuentes.
3. **Selección de variables**: de cada par de variables con |r| ≥ 0.95 se descarta una, porque aporta información redundante. La variable objetivo de E1 nunca se descarta.

In [ ]:
dias = crudo[crudo.fuente.eq("banxico")].shape[0]
meses = largo[largo.fuente.eq("banxico") & largo.origen.str.startswith("promedio")].shape[0]
print(f"Banxico: {dias:,} registros diarios -> {meses} registros mensuales ({1 - meses / max(dias, 1):.1%} menos)")
print(f"Ventana: {len(limpio)} meses en Plata -> {len(panel)} meses en Oro")
cand = [c for c in ["ingresos_colima", "personal_colima", "remuneraciones_colima", "inpc", "tiie", "tipo_cambio"] if c in limpio]
display(limpio.loc[config.FECHA_INICIO_ORO:, cand].corr().round(2))
print("Seleccionadas:", oro_info["variables_seleccionadas"])
display(pd.DataFrame(oro_info["variables_descartadas"]))

## 8. Corrida completa (lo mismo que ejecuta GitHub Actions) y salidas en Oro

In [ ]:
reporte = pipeline.correr(BRONCE, SALIDA)
for carpeta in ["plata", "oro", "cuarentena", "reportes"]:
    print(carpeta, "->", sorted(p.name for p in getattr(R, carpeta).iterdir()))
print(json.dumps(reporte["puerta_de_calidad"], indent=1, ensure_ascii=False))

## 9. Conclusiones

> **[COMPLETAR con los datos reales]**: tasa de aprobación de la Puerta, qué series tuvieron faltantes y con qué mecanismo, qué atípicos se trataron y cuáles se conservaron, y qué variables quedaron para E2 y E3.